In [ ]:
pip install faiss-cpu sentence-transformers pandas

In [2]:
import pandas as pd

# Your dataset (question | answer)
df = pd.read_csv("/content/drive/MyDrive/ai/normalized_ds_questions.csv")

questions = df["question"].tolist()
answers = df["answer"].tolist()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from sentence_transformers import SentenceTransformer

# Lightweight embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

question_embeddings = embed_model.encode(questions, normalize_embeddings=True)

In [6]:
import faiss
import numpy as np

dimension = question_embeddings.shape[1]

# index = faiss.IndexFlatL2(dimension)
index = faiss.IndexFlatIP(dimension)
index.add(np.array(question_embeddings))

print("FAISS index ready:", index.ntotal)

FAISS index ready: 1326


In [ ]:
# This step loads the Mistral model and defines a reusable function to generate responses from prompts.
# It ensures controlled and consistent output generation using deterministic settings.

from transformers import AutoTokenizer, AutoModelForCausalLM  # Hugging Face model utilities
import torch  # PyTorch backend

# Load model and tokenizer
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token to avoid warnings
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use half precision for efficiency
    device_map="auto"  # Automatically assign device (GPU/CPU)
)

def call_llm(prompt):
    # Tokenize input prompt and move to model device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate output with controlled settings
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,   # Limit response length
        temperature=0.1,      # Low randomness
        do_sample=False,      # Deterministic output
    )

    # Decode and clean output
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output.replace(prompt, "").strip()

In [ ]:
# This function extracts JSON from LLM output using regex and converts it into a dictionary.
# It ensures robust parsing and provides default values if extraction fails.

import json
import re

def parse_output(output):
    try:
        # Extract JSON block using regex
        json_str = re.search(r'\{.*\}', output, re.DOTALL).group()

        # Convert JSON string to dictionary
        return json.loads(json_str)

    except:
        # Return fallback response if parsing fails
        return {
            "score": 0,
            "confidence": 0.0,
            "needs_retrieval": True,
            "reason": "Parsing failed"
        }

In [ ]:
# This function safely extracts and parses JSON output from the LLM response.
# It prevents crashes by returning a default fallback if parsing fails.

def safe_parse_output(output):
    try:
        # Extract JSON substring from model output
        start = output.find("{")
        end = output.rfind("}") + 1

        # Convert JSON string to Python dictionary
        return json.loads(output[start:end])

    except:
        # Return default values if parsing fails
        return {
            "score": 0,
            "reason": "Parsing failed",
            "needs_retrieval": True
        }

In [ ]:
# This function uses an LLM to decide whether retrieval is needed based on answer quality.
# It returns a parsed JSON decision indicating score and reasoning.

def agent_decision(query, answer):
    # Build decision prompt for LLM
    prompt = f"""
You are a strict evaluator.
Return ONLY valid JSON.
No explanation.
Decide if retrieval is needed.

Rules:
- If answer is incorrect → true
- If answer is incomplete → true
- If answer is gibberish → true
- If answer is factually wrong → true
- If answer lacks key concepts → true
- If answer is vague → true
- If answer is correct and complete → false
- Otherwise → false

Now evaluate:

Q: {query}
A: {answer}
Return ONLY JSON:
{{
  "score": number,
  "reason": "short explanation"
}}
"""

    # Call LLM for decision
    output = call_llm(prompt)

    return output

In [ ]:
# This function retrieves the top-k most relevant question-answer pairs using FAISS similarity search.
# It encodes the query, finds nearest vectors, and maps them back to original data.

def retrieve(query, k=3):
    # Convert query into normalized embedding
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search for top-k similar embeddings in FAISS index
    distances, indices = index.search(query_embedding, k)

    # Collect corresponding question-answer pairs
    results = []
    for i in indices[0]:
        results.append((questions[i], answers[i]))

    return results

In [ ]:
# This function evaluates the user's answer using an LLM with strict scoring rules.
# It optionally uses retrieved context and returns a concise score with a reason.

def evaluate_answer(query, answer, context=None):

    # Prepare context text if available
    context_text = ""
    if context:
        context_text = "\n".join([c[1] for c in context])

    # Build strict evaluation prompt
    prompt  = f"""
You are a strict evaluator.

DO NOT explain.
DO NOT repeat context.
DO NOT teach.

Your ONLY job is to score the answer.

Rules:
- Gibberish → 0
- Wrong → 0–2
- Partial → 3–5
- Basic correct → 6–7
- Detailed → 8–9
- Perfect → 10

Return EXACTLY in this format:
Score: <number>
Reason: <short sentence>

If you output anything else, it is WRONG.

Question: {query}
Answer: {answer}
"""

    # Call LLM to generate evaluation
    output = call_llm(prompt)

    return output

In [ ]:
# This function defines the Agent AI pipeline that decides whether retrieval is needed and evaluates the answer accordingly.
# It combines decision-making, optional context retrieval, and final evaluation into a structured output.

def agent_pipeline(query, answer):

    # Step 1: Decide if retrieval is needed
    decision = agent_decision(query, answer)

    # Step 2: If retrieval is required
    if decision.get("needs_retrieval", True):
        # Retrieve relevant context
        context = retrieve(query)

        # Evaluate answer using retrieved context
        evaluation = evaluate_answer(
            query,
            answer,
            context=context
        )

        return {
            "retrieval_used": True,
            "decision": decision,
            "context": context,
            "evaluation": evaluation
        }

    else:
        # Evaluate answer without retrieval
        evaluation = evaluate_answer(query, answer)

        return {
            "retrieval_used": False,
            "decision": decision,
            "context": None,
            "evaluation": evaluation
        }

In [ ]:
# This step runs the full Agent AI pipeline by selecting a random question and evaluating the user’s answer.
# It integrates decision-making, retrieval, and evaluation into one interactive flow.

querys = df["question"].tolist()  # Extract all questions

import random  # For random selection

# Pick a random question
query = random.choice(querys)

# Display the question
print("Question:", query)

# Take user input as answer
answer = input("\nYour Answer: ")

# Run full agent pipeline and print result
print(agent_pipeline(query, answer))

Question: What is a live lock, and how does it differ from a deadlock?

Your Answer: In Livelock, threads are running, active, and changing states (e.g., locking/releasing) but stuck in a loop. In Deadlock, threads are waiting/blocked and not executing.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{'retrieval_used': True, 'decision': {'score': True, 'reason': 'incomplete'}, 'context': [('What is a live lock, and how does it differ from a deadlock?', '1. Live Lock Occurs when two or more transactions keep responding to each others changes, but no progress is made.Unlike a deadlock, the transactions are not blocked; they are actively running, but they cannot complete. 2.Deadlock Adeadlockoccurs when two or more transactions are waiting on each others resources indefinitely, blocking all progress.No progress can be made unless one of the transactions is terminated'), ('What is a deadlock in DBMS? How can it be prevented?', 'Adeadlockoccurs when two or more transactions are blocked because each transaction is waiting for the other to release resources. This results in a situation where none of the transactions can proceed. Prevention techniques: Lock ordering:Ensuring that all transactions acquire locks in the same predefined order.Timeouts:Automatically rolling back transactions th